# Stack your own inlist on a POSYDON scenario

<div class='alert alert-info'>

In this how-to you take the POSYDON v2.0 physics of an HMS-HMS grid and
change a few MESA settings on top of it: you stop every run after at most
5000 models (`max_model_number`), and lower the age limit. Everything
else stays as in the release.

</div>

You do not copy and edit the POSYDON inlists. You write a small inlist
with only the keys you want to change, and list it in the `ini` file's
`[user_inlists]` section. `posydon-grid` stacks it on top of the
release (and rerun), key by key: your keys win, every other key keeps the
release's value.

<div class='alert alert-info'>

The outputs on this page were made with a stand-in for MESA, so the
compilation is instant, and the paths were shortened to `/home/user/...`.
With your own MESA, `posydon-grid setup` compiles MESA, which takes a few
minutes, and the paths are yours.

</div>

## Before you start

You need the same things as in
[Set up an HMS-HMS grid with a POSYDON rerun](hms_hms_rerun.ipynb#Before-you-start):

- POSYDON installed, with `posydon-grid` on your `PATH`;
- `MESA_DIR` and `MESASDK_ROOT` set, and the MESA SDK loaded, for the
  POSYDON version of MESA r11701;
- a clone of POSYDON-MESA-INLISTS on its `v2_sources` branch in
  `~/POSYDON-MESA-INLISTS`.

Make a new, empty folder for this grid and start Jupyter (or a terminal)
there. The cells below run in that folder; a line starting with `!` is a
shell command, which you can also type in a terminal without the `!`.

## 1. Write your inlist

A user inlist is an ordinary MESA inlist, with only the keys you change.
`max_model_number` and `max_age` are star controls, so they go in
`&controls`. Here we also set something in `&star_job`, to show that
both namelists can be in one file:

In [1]:
%%writefile my_limits.inlist
&star_job
    pgstar_flag = .false.
/

&controls
    max_model_number = 5000
    max_age = 1d10
/

Writing my_limits.inlist


Each key goes into the namelist it is written in: `pgstar_flag` into
`&star_job` and the other two into `&controls`.

<div class='alert alert-warning'>

MESA stops with a namelist error, when the run starts, at a key it does
not know. `posydon-grid explain` (step 3) and `posydon-grid setup` warn
about every key of your files that is not a key of the namelist it goes
into, such as a misspelled `max_modle_number` in `&controls`, or
`max_model_number` in `&star_job`. The warning names the key, the namelist
and the file; it does not stop the setup. The keys are those of MESA's
defaults of the release's MESA installation, in
`~/POSYDON-MESA-INLISTS/r11701/mesa/default/defaults/` for r11701
(`star/controls.defaults`, `star/star_job.defaults`,
`binary/binary_controls.defaults`, `binary/binary_job.defaults`). Read the
warnings before you submit the grid.

</div>

## 2. Add it to the ini file

`star1` and `star2` take an inlist for that star; the same file can go to
both. Make a few grid points as well:

In [2]:
import csv

with open("grid.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["initial_z", "Zbase", "m1", "m2",
                     "initial_period_in_days"])
    writer.writerow([0.0142, 0.0142, 30.0, 21.0, 10.0])

In [3]:
%%writefile HMS-HMS.ini
[grid]
type = HMS-HMS
mode = fixed
points = grid.csv
run_directory = run
zams = ~/POSYDON-MESA-INLISTS/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data

[scenario]
release = v2.0
inlist_repo = ~/POSYDON-MESA-INLISTS

[user_inlists]
star1 = my_limits.inlist
star2 = my_limits.inlist

[output]
history_interval = 1
star1 = history, final_model
star2 = history, final_model
binary = history

[submission]
backend = slurm_array
partition = my_partition
account = my_account
walltime = 2-00:00:00
cpus_per_task = 4

Writing HMS-HMS.ini


<details>
<summary><strong>All the places an inlist can go</strong></summary>

| Key in `[user_inlists]` | Goes into |
|---|---|
| `binary` | `&binary_controls` and `&binary_job` |
| `binary_controls`, `binary_job` | only that namelist |
| `star1`, `star2` | `&controls` and `&star_job` of that star |
| `star1_controls`, `star1_job`, `star2_controls`, `star2_job` | only that namelist of that star |
| `star1_formation_step0`, ... | one formation step of star 1 |

With a single-namelist key, a file that holds both namelists gives only
that one. Several files for one key are separated by commas and stacked
in order, the last winning: `star1 = my_limits.inlist, my_winds.inlist`.

In a single-star grid (single_HMS, single_HeMS), which runs star 1 as its
formation steps, `star1` goes into every step.

`[user_inlists]` also takes whole files, which replace the release's:
`history_columns`, `binary_history_columns`, `profile_columns`, and the
Fortran sources `run_star_extras`, `run_binary_extras`,
`run_star1_extras` and `run_star2_extras`.

</details>

## 3. Check the stacking

`posydon-grid explain` now shows your file as the top layer,
`[user_inlists]`, for both stars:

In [4]:
!posydon-grid explain HMS-HMS.ini > explain.txt
!grep -A1 '^  max_model_number ' explain.txt

  max_model_number = 5000  [user_inlists] /home/user/grids/stack_own_inlist/my_limits.inlist
      over -1  [mesa_inlists] r11701/mesa/default/defaults/star/controls.defaults
--
  max_model_number = 5000  [user_inlists] /home/user/grids/stack_own_inlist/my_limits.inlist
      over -1  [mesa_inlists] r11701/mesa/default/defaults/star/controls.defaults


`max_model_number` was MESA's default (`-1`, no limit) and is now 5000.
`max_age` shows the whole stack: your value over the release's over
MESA's default:

In [5]:
!grep -A2 '^  max_age ' explain.txt

  max_age = 1d10  [user_inlists] /home/user/grids/stack_own_inlist/my_limits.inlist
      over 1.4e10  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist1
      over 1d36  [mesa_inlists] r11701/mesa/default/defaults/star/controls.defaults
--
  max_age = 1d10  [user_inlists] /home/user/grids/stack_own_inlist/my_limits.inlist
      over 1.4e10  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist2
      over 1d36  [mesa_inlists] r11701/mesa/default/defaults/star/controls.defaults


Your keys win because `[user_inlists]` is stacked last. From bottom to
top: MESA's defaults, the release (and rerun), your inlists, and then the
keys the setup writes itself (the ZAMS model, the output options and the
wiring between the inlists). So you can change any MESA key except
those last few.

## 4. Set up

In [6]:
!posydon-grid setup HMS-HMS.ini
!grep -n 'max_model_number\|max_age \|pgstar_flag' run/binary/inlist1

81:	max_model_number = 5000
84:	max_age = 1d10
86:	num_adjusted_dt_steps_before_max_age = 0
87:	dt_years_for_steps_before_max_age = 1d6
1677:	pgstar_flag = .false.


The record lists your file as the last one stacked into each namelist:

In [7]:
!sed -n '/^\[inlists\]/,/^$/p' run/setup_record.ini | grep -A4 'inlist1 &controls'

binary/inlist1 &controls = 
	mesa_inlists     r11701/mesa/default/defaults/star/controls.defaults
	posydon_inlists  r11701/releases/v2.0/common/binary/inlist1
	user_inlists     /home/user/grids/stack_own_inlist/my_limits.inlist
binary/inlist1 &star_job = 


## Variations

- **On top of a rerun.** Add `rerun = <name>` to `[scenario]`: your
  inlist then goes on top of the rerun's changes.
- **The binary.** Put `&binary_controls` keys (for example
  `max_tries_to_achieve`) in a file and list it as `binary = <file>`, or
  as `binary_controls = <file>`.
- **Your own Fortran.** `run_star_extras = my_run_star_extras.f` in
  `[user_inlists]` uses your file instead of the release's, for every
  star. It can have any name; it is copied as `run_star_extras.f`.
- **A full example**, with a rerun, inlists for the binary and both stars,
  a Fortran source and a column list:
  [HMS-HMS_rerun_and_my_physics.ini](https://github.com/POSYDON-code/POSYDON/tree/main/docs/_source/components-overview/mesa_grids/examples/HMS-HMS_rerun_and_my_physics.ini).